<a href="https://colab.research.google.com/github/hishanthp2008-del/DAA/blob/main/DAA2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import random
import string


# ---------------------------------------------------------
# 1. NAIVE STRING MATCHING
# ---------------------------------------------------------
def naive_search(text, pattern):
    n = len(text)
    m = len(pattern)

    comparisons = 0
    matches = []

    for i in range(n - m + 1):
        j = 0

        while j < m:
            comparisons += 1

            if text[i + j] != pattern[j]:
                break

            j += 1

        if j == m:
            matches.append(i)

    return matches, comparisons


# ---------------------------------------------------------
# 2. RABIN-KARP STRING MATCHING
# ---------------------------------------------------------
def rabin_karp_search(text, pattern):
    n = len(text)
    m = len(pattern)

    if m > n:
        return [], 0

    # Base and prime modulus
    d = 256
    q = 101

    comparisons = 0
    matches = []

    # h = d^(m-1) % q
    h = 1
    for _ in range(m - 1):
        h = (h * d) % q

    pattern_hash = 0
    text_hash = 0

    # Calculate initial hash values
    for i in range(m):
        pattern_hash = (d * pattern_hash + ord(pattern[i])) % q
        text_hash = (d * text_hash + ord(text[i])) % q

    for i in range(n - m + 1):

        # Compare characters only when hash values match
        if pattern_hash == text_hash:

            j = 0

            while j < m:
                comparisons += 1

                if text[i + j] != pattern[j]:
                    break

                j += 1

            if j == m:
                matches.append(i)

        # Calculate hash for next window
        if i < n - m:
            text_hash = (
                d * (text_hash - ord(text[i]) * h)
                + ord(text[i + m])
            ) % q

            if text_hash < 0:
                text_hash += q

    return matches, comparisons


# ---------------------------------------------------------
# 3. KMP STRING MATCHING
# ---------------------------------------------------------
def compute_lps(pattern):
    """
    Compute the Longest Prefix Suffix (LPS) array.
    Comparisons made during LPS construction are counted.
    """
    m = len(pattern)
    lps = [0] * m

    length = 0
    i = 1
    comparisons = 0

    while i < m:
        comparisons += 1

        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1
        else:
            if length != 0:
                length = lps[length - 1]
            else:
                lps[i] = 0
                i += 1

    return lps, comparisons


def kmp_search(text, pattern):
    n = len(text)
    m = len(pattern)

    if m == 0:
        return [], 0

    # Build LPS array
    lps, preprocessing_comparisons = compute_lps(pattern)

    matches = []
    comparisons = 0

    i = 0  # index for text
    j = 0  # index for pattern

    while i < n:

        comparisons += 1

        if text[i] == pattern[j]:
            i += 1
            j += 1

            if j == m:
                matches.append(i - j)
                j = lps[j - 1]

        else:
            if j != 0:
                j = lps[j - 1]
            else:
                i += 1

    # Total comparisons includes LPS preprocessing
    total_comparisons = preprocessing_comparisons + comparisons

    return matches, total_comparisons


# ---------------------------------------------------------
# GENERATE TEXT OF 10,000 CHARACTERS
# ---------------------------------------------------------
def generate_text(length=10000):
    characters = string.ascii_lowercase + " "
    return ''.join(random.choice(characters) for _ in range(length))


# ---------------------------------------------------------
# GENERATE PATTERNS
# ---------------------------------------------------------
def generate_patterns(text):
    """
    Use substrings from the text as patterns so that
    each pattern is guaranteed to occur at least once.
    """
    lengths = [5, 10, 20, 50]

    patterns = {}

    for length in lengths:
        start = random.randint(0, len(text) - length)
        patterns[length] = text[start:start + length]

    return patterns


# ---------------------------------------------------------
# MAIN PROGRAM
# ---------------------------------------------------------
def main():

    # Generate text with exactly 10,000 characters
    text = generate_text(10000)

    # Generate patterns of different lengths
    patterns = generate_patterns(text)

    print("=" * 75)
    print("COMPARATIVE ANALYSIS OF STRING MATCHING ALGORITHMS")
    print("=" * 75)

    print("\nText length:", len(text))
    print("Pattern lengths:", list(patterns.keys()))

    print("\n" + "-" * 75)
    print(f"{'Pattern':<12}{'Naive':>15}{'Rabin-Karp':>18}{'KMP':>15}")
    print("-" * 75)

    results = {}

    for length, pattern in patterns.items():

        # Naive
        naive_matches, naive_comparisons = naive_search(
            text, pattern
        )

        # Rabin-Karp
        rk_matches, rk_comparisons = rabin_karp_search(
            text, pattern
        )

        # KMP
        kmp_matches, kmp_comparisons = kmp_search(
            text, pattern
        )

        results[length] = {
            "Naive": naive_comparisons,
            "Rabin-Karp": rk_comparisons,
            "KMP": kmp_comparisons
        }

        print(
            f"{length:<12}"
            f"{naive_comparisons:>15}"
            f"{rk_comparisons:>18}"
            f"{kmp_comparisons:>15}"
        )

    print("-" * 75)

    # -----------------------------------------------------
    # Display number of matches
    # -----------------------------------------------------
    print("\nNUMBER OF MATCHES FOUND")
    print("-" * 50)
    print(f"{'Pattern Length':<20}{'Matches':>15}")
    print("-" * 50)

    for length, pattern in patterns.items():
        matches, _ = naive_search(text, pattern)

        print(f"{length:<20}{len(matches):>15}")

    # -----------------------------------------------------
    # Display detailed comparison
    # -----------------------------------------------------
    print("\n\nANALYSIS")
    print("=" * 75)

    for length in [5, 10, 20, 50]:

        naive = results[length]["Naive"]
        rk = results[length]["Rabin-Karp"]
        kmp = results[length]["KMP"]

        print(f"\nPattern Length = {length}")
        print(f"  Naive String Matching : {naive} comparisons")
        print(f"  Rabin-Karp            : {rk} comparisons")
        print(f"  KMP                   : {kmp} comparisons")

        minimum = min(naive, rk, kmp)

        if minimum == naive:
            print("  Best: Naive")
        elif minimum == rk:
            print("  Best: Rabin-Karp")
        else:
            print("  Best: KMP")


# Run the program
if __name__ == "__main__":
    main()


COMPARATIVE ANALYSIS OF STRING MATCHING ALGORITHMS

Text length: 10000
Pattern lengths: [5, 10, 20, 50]

---------------------------------------------------------------------------
Pattern               Naive        Rabin-Karp            KMP
---------------------------------------------------------------------------
5                     10387               108          10368
10                    10376               117          10372
20                    10385               132          10393
50                    10370               168          10412
---------------------------------------------------------------------------

NUMBER OF MATCHES FOUND
--------------------------------------------------
Pattern Length              Matches
--------------------------------------------------
5                                 1
10                                1
20                                1
50                                1


ANALYSIS

Pattern Length = 5
  Naive String Matching 